# Tratamento dos Dados (Silver)

## Objetivos
- Padronizar nomes das colunas para snake_case
- Tratar valores nulos e inconsistentes
- Converter tipos de dados apropriadamente
- Extrair colunas úteis para análises gráficas
- Preparar dados para visualizações (pizza, regressão linear, boxplot)

## Estrutura dos Dados Originais
O dataset contém informações sobre escolas brasileiras do INEP com 19 colunas principais:
- Informações geográficas (UF, Município, Localização, Latitude, Longitude)
- Características administrativas (Dependência, Categoria, Porte)
- Informações educacionais (Etapas de Ensino, Modalidades)
- Status operacional (Restrição de Atendimento, Regulamentação)

In [25]:
# === IMPORTS ===
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, split, size, coalesce, lit, regexp_replace, lower, regexp_extract
from pyspark.sql.types import IntegerType, DoubleType, StringType
import pyspark.sql.functions as F
import os
from pathlib import Path


In [26]:
# === INICIALIZAÇÃO DO SPARK ===
spark = (SparkSession.builder
    .appName("inep_schools_analysis")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate())

print("✓ Spark Session inicializada")


✓ Spark Session inicializada


In [27]:
# === CONFIGURAÇÃO DO CAMINHO DOS DADOS BRONZE ===
bronze_env_path = os.getenv("BRONZE_DATA_PATH", "../data-layer/bronze/escolas_inep.csv")
bronze_csv_path = Path(bronze_env_path)
if not bronze_csv_path.is_absolute():
    bronze_csv_path = (Path.cwd() / bronze_csv_path).resolve()

print("=== UTILIZANDO ARQUIVO BRONZE ===")
print(bronze_csv_path)


=== UTILIZANDO ARQUIVO BRONZE ===
/home/lucascaldasb/Documentos/grupo-18-bancos-2/inep-schools-analysis/data-layer/bronze/escolas_inep.csv


In [28]:
# === CARREGAMENTO DOS DADOS BRONZE ===
df = spark.read.csv(str(bronze_csv_path), header=True, sep=",")

print("=== ESTRUTURA INICIAL DOS DADOS ===")
print(f"Total de registros: {df.count():,}")
print(f"Número de colunas: {len(df.columns)}")
df.printSchema()


=== ESTRUTURA INICIAL DOS DADOS ===
Total de registros: 212,386
Número de colunas: 19
root
 |-- Restrição de Atendimento: string (nullable = true)
 |-- Escola: string (nullable = true)
 |-- Código INEP: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- Município: string (nullable = true)
 |-- Localização: string (nullable = true)
 |-- Localidade Diferenciada: string (nullable = true)
 |-- Categoria Administrativa: string (nullable = true)
 |-- Endereço: string (nullable = true)
 |-- Telefone: string (nullable = true)
 |-- Dependência Administrativa: string (nullable = true)
 |-- Categoria Escola Privada: string (nullable = true)
 |-- Conveniada Poder Público: string (nullable = true)
 |-- Regulamentação pelo Conselho de Educação: string (nullable = true)
 |-- Porte da Escola: string (nullable = true)
 |-- Etapas e Modalidade de Ensino Oferecidas: string (nullable = true)
 |-- Outras Ofertas Educacionais: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-

In [29]:
# === ANÁLISE DE VALORES NULOS ===
print("=== ANÁLISE DE VALORES NULOS ===")
null_counts = df.select([F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c) for c in df.columns]).collect()[0]
for col_name in df.columns:
    null_count = null_counts[col_name]
    if null_count > 0:
        print(f"{col_name}: {null_count:,} valores nulos/vazios")


=== ANÁLISE DE VALORES NULOS ===
Telefone: 42,433 valores nulos/vazios
Regulamentação pelo Conselho de Educação: 31,321 valores nulos/vazios
Porte da Escola: 31,321 valores nulos/vazios
Etapas e Modalidade de Ensino Oferecidas: 33,163 valores nulos/vazios
Outras Ofertas Educacionais: 153,326 valores nulos/vazios


In [30]:
# === RENOMEAÇÃO DE COLUNAS PARA SNAKE_CASE ===
df_clean = df.withColumnRenamed("Restrição de Atendimento", "restricao_atendimento") \
             .withColumnRenamed("Escola", "nome_escola") \
             .withColumnRenamed("Código INEP", "codigo_inep") \
             .withColumnRenamed("UF", "uf") \
             .withColumnRenamed("Município", "municipio") \
             .withColumnRenamed("Localização", "localizacao") \
             .withColumnRenamed("Localidade Diferenciada", "localidade_diferenciada") \
             .withColumnRenamed("Categoria Administrativa", "categoria_administrativa") \
             .withColumnRenamed("Endereço", "endereco") \
             .withColumnRenamed("Telefone", "telefone") \
             .withColumnRenamed("Dependência Administrativa", "dependencia_administrativa") \
             .withColumnRenamed("Categoria Escola Privada", "categoria_escola_privada") \
             .withColumnRenamed("Conveniada Poder Público", "conveniada_poder_publico") \
             .withColumnRenamed("Regulamentação pelo Conselho de Educação", "regulamentacao_conselho") \
             .withColumnRenamed("Porte da Escola", "porte_escola") \
             .withColumnRenamed("Etapas e Modalidade de Ensino Oferecidas", "etapas_modalidades") \
             .withColumnRenamed("Outras Ofertas Educacionais", "outras_ofertas") \
             .withColumnRenamed("Latitude", "latitude") \
             .withColumnRenamed("Longitude", "longitude")

print("=== COLUNAS RENOMEADAS PARA SNAKE_CASE ===")
df_clean.printSchema()


=== COLUNAS RENOMEADAS PARA SNAKE_CASE ===
root
 |-- restricao_atendimento: string (nullable = true)
 |-- nome_escola: string (nullable = true)
 |-- codigo_inep: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- localizacao: string (nullable = true)
 |-- localidade_diferenciada: string (nullable = true)
 |-- categoria_administrativa: string (nullable = true)
 |-- endereco: string (nullable = true)
 |-- telefone: string (nullable = true)
 |-- dependencia_administrativa: string (nullable = true)
 |-- categoria_escola_privada: string (nullable = true)
 |-- conveniada_poder_publico: string (nullable = true)
 |-- regulamentacao_conselho: string (nullable = true)
 |-- porte_escola: string (nullable = true)
 |-- etapas_modalidades: string (nullable = true)
 |-- outras_ofertas: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)



In [31]:
# === CONVERSÃO DE COORDENADAS PARA DOUBLE ===
df_clean = df_clean.withColumn("latitude", 
    when(col("latitude").rlike("^-?\\d+\\.?\\d*$"), col("latitude").cast(DoubleType()))
    .otherwise(None)
).withColumn("longitude", 
    when(col("longitude").rlike("^-?\\d+\\.?\\d*$"), col("longitude").cast(DoubleType()))
    .otherwise(None)
)

print("✓ Coordenadas convertidas para DoubleType")


✓ Coordenadas convertidas para DoubleType


In [32]:
# === LIMPEZA DE STRINGS (REMOVER ESPAÇOS EXTRAS) ===
string_columns = ["restricao_atendimento", "nome_escola", "uf", "municipio", 
                 "localizacao", "categoria_administrativa", "dependencia_administrativa",
                 "porte_escola", "etapas_modalidades"]

for col_name in string_columns:
    df_clean = df_clean.withColumn(col_name, trim(col(col_name)))

print("✓ Strings limpas (espaços extras removidos)")


✓ Strings limpas (espaços extras removidos)


In [33]:
# === TRATAMENTO DE VALORES "NÃO INFORMADO" ===
df_clean = df_clean.withColumn("categoria_escola_privada", 
    when(col("categoria_escola_privada") == "Não Informado", None)
    .otherwise(col("categoria_escola_privada"))
)

print("✓ Valores 'Não Informado' convertidos para NULL")


✓ Valores 'Não Informado' convertidos para NULL


In [34]:
# === FILTRAGEM: APENAS ESCOLAS COM COORDENADAS VÁLIDAS ===
df_clean = df_clean.filter(
    col("latitude").isNotNull() & 
    col("longitude").isNotNull() &
    (col("latitude") != 0) & 
    (col("longitude") != 0)
)

print(f"=== DADOS APÓS LIMPEZA ===")
print(f"Registros válidos: {df_clean.count():,}")
print(f"Registros removidos: {df.count() - df_clean.count():,}")


=== DADOS APÓS LIMPEZA ===
Registros válidos: 156,423
Registros removidos: 55,963


In [35]:
# === COLUNA DERIVADA: NÚMERO DE ETAPAS/MODALIDADES ===
df_clean = df_clean.withColumn("etapas_list", 
    split(regexp_replace(col("etapas_modalidades"), r",\\s*", ","), ",")
).withColumn("num_etapas", 
    coalesce(size(col("etapas_list")), lit(0))  # Substituir NULL por 0
)

print("✓ Coluna 'num_etapas' criada (NULLs substituídos por 0)")


✓ Coluna 'num_etapas' criada (NULLs substituídos por 0)


In [36]:
# === COLUNA DERIVADA: PORTE NUMÉRICO ===
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

def map_porte_to_numeric(porte):
    if porte is None:
        return None
    porte_lower = porte.lower().strip()
    if "pequeno" in porte_lower or "até 50" in porte_lower:
        return 1
    elif "médio" in porte_lower or "51" in porte_lower or "201" in porte_lower or "501" in porte_lower:
        return 2
    elif "grande" in porte_lower or "1000" in porte_lower:
        return 3
    else:
        return None

map_porte_udf = udf(map_porte_to_numeric, IntegerType())
df_clean = df_clean.withColumn("porte_numerico", map_porte_udf(col("porte_escola")))

print("✓ Coluna 'porte_numerico' criada (1=Pequeno, 2=Médio, 3=Grande)")


✓ Coluna 'porte_numerico' criada (1=Pequeno, 2=Médio, 3=Grande)


In [37]:
# === COLUNA DERIVADA: FLAG ESCOLAS RURAIS ===
df_clean = df_clean.withColumn("is_rural", 
    when(col("localizacao") == "Rural", 1).otherwise(0)
)

print("✓ Coluna 'is_rural' criada (1=Rural, 0=Urbana)")


✓ Coluna 'is_rural' criada (1=Rural, 0=Urbana)


In [38]:
# === COLUNA DERIVADA: FLAG ESCOLAS PÚBLICAS ===
df_clean = df_clean.withColumn("is_publica", 
    when(col("dependencia_administrativa").isin(["Estadual", "Municipal", "Federal"]), 1)
    .otherwise(0)
)

print("✓ Coluna 'is_publica' criada (1=Pública, 0=Privada)")


✓ Coluna 'is_publica' criada (1=Pública, 0=Privada)


In [39]:
# === COLUNA DERIVADA: REGIÃO DO BRASIL ===
def get_regiao(uf):
    regioes = {
        'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
        'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 
        'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
        'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
        'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
        'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
    }
    return regioes.get(uf, 'Outro')

get_regiao_udf = udf(get_regiao, StringType())
df_clean = df_clean.withColumn("regiao", get_regiao_udf(col("uf")))

print("✓ Coluna 'regiao' criada")


✓ Coluna 'regiao' criada


In [40]:
# === RESUMO DAS COLUNAS DERIVADAS ===
print("=== COLUNAS DERIVADAS CRIADAS ===")
print("Colunas adicionadas:")
print("- num_etapas: número de etapas/modalidades oferecidas")
print("- porte_numerico: porte da escola (1=Pequeno, 2=Médio, 3=Grande)")
print("- is_rural: flag para escolas rurais (1=Rural, 0=Urbana)")
print("- is_publica: flag para escolas públicas (1=Pública, 0=Privada)")
print("- regiao: região do Brasil baseada na UF")


=== COLUNAS DERIVADAS CRIADAS ===
Colunas adicionadas:
- num_etapas: número de etapas/modalidades oferecidas
- porte_numerico: porte da escola (1=Pequeno, 2=Médio, 3=Grande)
- is_rural: flag para escolas rurais (1=Rural, 0=Urbana)
- is_publica: flag para escolas públicas (1=Pública, 0=Privada)
- regiao: região do Brasil baseada na UF


In [41]:
# === DEFINIÇÃO DAS COLUNAS PARA ANÁLISE ===
colunas_analise = [
    "codigo_inep", "nome_escola", "uf", "municipio", "regiao",
    "localizacao", "is_rural", "dependencia_administrativa", "is_publica",
    "porte_escola", "porte_numerico", "etapas_modalidades", "num_etapas",
    "latitude", "longitude", "restricao_atendimento"
]

print("✓ Colunas para análise definidas")


✓ Colunas para análise definidas


In [42]:
# === SELEÇÃO FINAL DAS COLUNAS ===
df_final = df_clean.select(*colunas_analise)

print("=== DADOS FINAIS PARA ANÁLISE ===")
print(f"Total de registros: {df_final.count():,}")
print(f"Colunas selecionadas: {len(colunas_analise)}")
print()
print("Colunas incluídas:")
for col in colunas_analise:
    print(f"- {col}")


=== DADOS FINAIS PARA ANÁLISE ===
Total de registros: 156,423
Colunas selecionadas: 16

Colunas incluídas:
- codigo_inep
- nome_escola
- uf
- municipio
- regiao
- localizacao
- is_rural
- dependencia_administrativa
- is_publica
- porte_escola
- porte_numerico
- etapas_modalidades
- num_etapas
- latitude
- longitude
- restricao_atendimento


In [43]:
# === CONFIGURAÇÃO DE CONEXÃO COM POSTGRESQL ===
postgres_host = os.getenv("POSTGRES_HOST", os.getenv("DB_HOST", "localhost"))
postgres_port = os.getenv("POSTGRES_PORT", os.getenv("DB_PORT", "5432"))
postgres_db = os.getenv("POSTGRES_DB", "inep_db")
postgres_user = os.getenv("POSTGRES_USER", "inep")
postgres_password = os.getenv("POSTGRES_PASSWORD", "inep")
postgres_table = os.getenv("POSTGRES_TABLE", "escolas")

jdbc_url = f"jdbc:postgresql://{postgres_host}:{postgres_port}/{postgres_db}"
jdbc_properties = {
    "user": postgres_user,
    "password": postgres_password,
    "driver": "org.postgresql.Driver"
}

print("=== CONFIGURAÇÃO POSTGRESQL ===")
print(f"Host: {postgres_host}")
print(f"Port: {postgres_port}")
print(f"Database: {postgres_db}")
print(f"Table: {postgres_table}")


=== CONFIGURAÇÃO POSTGRESQL ===
Host: localhost
Port: 5432
Database: inep_db
Table: escolas


In [44]:
# === LIMPEZA DA TABELA POSTGRESQL ===
print("=== LIMPANDO TABELA POSTGRESQL ===")
print(f"Tabela: {postgres_table}")

(df_final.limit(0).write
    .mode("overwrite")
    .option("truncate", "true")
    .jdbc(url=jdbc_url, table=postgres_table, properties=jdbc_properties))

print("✓ Tabela limpa (truncate executado)")


=== LIMPANDO TABELA POSTGRESQL ===
Tabela: escolas
✓ Tabela limpa (truncate executado)


In [45]:
# === COLETA DOS DADOS PARA INSERÇÃO ===
print("=== COLETANDO DADOS DO DATAFRAME ===")
total_records = df_final.count()
rows = df_final.collect()

print(f"✓ {total_records:,} registros coletados para inserção")


=== COLETANDO DADOS DO DATAFRAME ===


✓ 156,423 registros coletados para inserção


In [46]:
# === VERIFICAÇÃO E IMPORTAÇÃO DO PSYCOPG2 ===
try:
    import psycopg2
    from psycopg2.extras import execute_values
    USE_PSYCOPG2 = True
    print("✓ psycopg2 encontrado - usando inserção otimizada")
except ImportError:
    print("⚠️  psycopg2 não encontrado. Usando Spark JDBC linha por linha (mais lento).")
    USE_PSYCOPG2 = False


✓ psycopg2 encontrado - usando inserção otimizada


In [47]:
# === INSERÇÃO NO POSTGRESQL (MÉTODO: PSYCOPG2) ===
if USE_PSYCOPG2:
    print("=== INSERINDO DADOS NO POSTGRESQL (LINHA POR LINHA) ===")
    print(f"Destino: {postgres_table} @ {postgres_host}:{postgres_port}/{postgres_db}")
    print(f"Total de registros: {total_records:,}")
    
    # Conectar ao PostgreSQL
    conn = psycopg2.connect(
        host=postgres_host,
        port=postgres_port,
        database=postgres_db,
        user=postgres_user,
        password=postgres_password
    )
    cur = conn.cursor()
    
    # Preparar query de inserção
    columns = ", ".join(colunas_analise)
    placeholders = ", ".join(["%s"] * len(colunas_analise))
    insert_query = f"INSERT INTO {postgres_table} ({columns}) VALUES ({placeholders})"
    
    # Inserir linha por linha
    inserted = 0
    for idx, row in enumerate(rows, 1):
        values = []
        for col in colunas_analise:
            val = row[col]
            # Tratar valores NULL para colunas NOT NULL
            if val is None:
                if col == "num_etapas":
                    val = 0  # Valor padrão para num_etapas
                elif col == "porte_numerico":
                    val = None  # Pode ser NULL
                else:
                    val = None
            values.append(val)
        
        cur.execute(insert_query, values)
        inserted += 1
        
        # Commit a cada 100 registros para melhor performance
        if inserted % 100 == 0:
            conn.commit()
            print(f"  Linha {inserted:,}/{total_records:,} inserida...", end='\r')
    
    # Commit final
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"\n✓ Dados gravados com sucesso no banco PostgreSQL!")
    print(f"  Total: {inserted:,} registros inseridos linha por linha")


=== INSERINDO DADOS NO POSTGRESQL (LINHA POR LINHA) ===
Destino: escolas @ localhost:5432/inep_db
Total de registros: 156,423
  Linha 156,400/156,423 inserida...
✓ Dados gravados com sucesso no banco PostgreSQL!
  Total: 156,423 registros inseridos linha por linha


In [48]:
# === INSERÇÃO NO POSTGRESQL (MÉTODO: SPARK JDBC - FALLBACK) ===
if not USE_PSYCOPG2:
    print("=== INSERINDO DADOS NO POSTGRESQL (LINHA POR LINHA - SPARK JDBC) ===")
    print(f"Destino: {postgres_table} @ {postgres_host}:{postgres_port}/{postgres_db}")
    print(f"Total de registros: {total_records:,}")
    
    inserted = 0
    for idx, row in enumerate(rows, 1):
        # Criar lista de valores na ordem das colunas
        row_data = [tuple([row[col] for col in colunas_analise])]
        
        # Criar DataFrame com uma única linha usando o schema do df_final
        single_row_df = spark.createDataFrame(row_data, schema=df_final.schema)
        
        # Inserir linha no banco
        (single_row_df.write
            .mode("append")
            .jdbc(url=jdbc_url, table=postgres_table, properties=jdbc_properties))
        
        inserted += 1
        
        # Mostrar progresso a cada 100 registros
        if inserted % 100 == 0 or inserted == total_records:
            print(f"  Linha {inserted:,}/{total_records:,} inserida...", end='\r')
    
    print(f"\n✓ Dados gravados com sucesso no banco PostgreSQL!")
    print(f"  Total: {inserted:,} registros inseridos linha por linha")
